# Exploración analítica de diabetes — Pima Indians

**Propósito:** practicar un flujo completo de análisis exploratorio de datos (EDA): revisar calidad, justificar tratamiento de ausencias, normalizar variables y estudiar patrones univariados, bivariados y multivariados. No es un instrumento diagnóstico.

El conjunto tiene 768 registros y 9 columnas. Se refiere a mujeres Pima de 21 años o más y la etiqueta es el inicio de diabetes dentro de cinco años según la definición histórica del dataset. Consulta las fuentes y límites en `README.md`.

## Cómo ejecutar
Instala `requirements.txt`, selecciona el kernel del entorno `.venv` y ejecuta de arriba abajo. El notebook usa las funciones explicadas de `src/analisis.py`; las tablas y figuras se guardan en carpetas del proyecto.

## 1. Preparación y carga
Se fija el directorio del proyecto a partir de la ubicación de este notebook; se importa el flujo común. La primera ejecución descarga el CSV si no está en `data/raw/`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src' / 'analisis.py').exists():
    ROOT = ROOT.parent if (ROOT.parent / 'src' / 'analisis.py').exists() else ROOT
sys.path.insert(0, str(ROOT / 'src'))
import analisis as eda

raw = eda.obtain_data()
print(f'Forma del dataset: {raw.shape[0]} filas × {raw.shape[1]} columnas')
display(raw.head())
display(raw.dtypes.to_frame('tipo detectado'))

## 2. Auditoría de calidad antes de limpiar
Contamos nulos explícitos, ceros sospechosos y duplicados. En cinco mediciones corporales, un cero no representa una lectura fisiológica válida en este contexto. Se preservarán las filas y se documentará su sustitución por mediana.

In [ ]:
print('Nulos explícitos por columna:')
display(raw.isna().sum().rename('nulos').to_frame())
print('Ceros en variables que se interpretarán como ausencias:')
display(raw[eda.ZERO_MISSING].eq(0).sum().rename('ceros sospechosos').to_frame())
print(f'Duplicados exactos: {raw.duplicated().sum()}')

## 3. Limpieza con trazabilidad
Las cadenas/valores se convierten a numérico. El objetivo se transforma a categoría No/Sí. El conteo de embarazos queda entero. Los ceros imposibles se convierten en NaN y los atributos se imputan con su mediana. Se excluyen solo filas sin etiqueta válida, ya que sin resultado no se puede comparar grupos. No se borran outliers automáticamente.

In [ ]:
clean, audit = eda.prepare_data(raw)
display(pd.Series(audit, dtype='object').to_frame('auditoría'))
display(clean.head())
display(clean.dtypes.to_frame('tipo final'))
print('Nulos luego de la limpieza:', int(clean.isna().sum().sum()))

## 4. Univariado: tendencias y distribuciones
Media, mediana, desviación, rango intercuartílico y asimetría describen centro, dispersión y forma. Las barras e histogramas enseñan concentración, asimetría y desequilibrio de clase. Una acumulación en la mediana puede reflejar la imputación, no una medición real.

In [ ]:
summary = eda.save_univariate(clean)
display(summary.round(2))
display(clean['resultado_diabetes'].value_counts().rename('casos').to_frame().assign(porcentaje=lambda x: 100*x['casos']/len(clean)).round(1))
from IPython.display import Image, display as show_image
show_image(Image(filename=str(eda.FIGURES / '01_distribuciones.png')))

## 5. Bivariado: asociaciones con resultado
Comparamos distribuciones con cajas y bigotes y calculamos coeficientes Pearson (asociación lineal) y Spearman (asociación monotónica por rangos). La etiqueta es binaria codificada como 0/1; los coeficientes sirven como resumen de asociación, sin interpretación causal.

In [ ]:
medians, corr_target = eda.save_bivariate(clean)
display(corr_target.round(3))
display(medians.round(2))
show_image(Image(filename=str(eda.FIGURES / '02_boxplots_por_resultado.png')))

## 6. Multivariado: correlaciones, relaciones conjuntas y PCA
La matriz examina asociaciones entre todos los atributos numéricos. Pairplot compara combinaciones de variables seleccionadas y colorea por clase. PCA estandariza las columnas antes de resumir variación; la proyección facilita inspección visual, pero no es un modelo diagnóstico ni necesariamente separa clases.

In [ ]:
corr = eda.save_multivariate(clean)
outliers = eda.iqr_outliers(clean)
display(corr.round(2))
display(outliers.to_frame())
for filename in ['03_mapa_correlaciones.png', '04_relaciones_conjuntas.png', '05_pca_exploratorio.png']:
    print(filename)
    show_image(Image(filename=str(eda.FIGURES / filename)))

## 7. Conclusión y entrega
Interpreta el patrón más visible con sus medidas y figuras, menciona la prevalencia de cada clase y enlaza la evidencia con su limitación. La correlación no es causalidad. Las observaciones extremas deben revisarse con contexto, y los datos imputados añaden incertidumbre.

Para producir el informe completo reproducible y todas las salidas, ejecuta `python src/analisis.py` en la terminal de VS Code. El archivo `reports/informe_analitico.md` contiene valores calculados, tablas, conclusiones y limitaciones.